# Хронологическая валидация

В предыдущих экспериментах использовалось случайное разделение данных на Train и Test. Однако в реальной задаче модель обучается на прошлых кампаниях и применяется к будущим контактам.

Проверим качество Logistic Regression на более поздних наблюдениях и оценим, насколько результаты меняются между периодами.

Для этого используем исходный датасет `bank-additional-full.csv` из UCI, в котором сохранён хронологический порядок строк.


## 1. Загрузка данных

Загрузим исходный хронологический датасет UCI и `banking.csv`, который использовался в предыдущих экспериментах.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    StandardScaler
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

# Корневая папка проекта
project_root = Path.cwd().resolve()

if project_root.name == 'notebooks':
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preprocessing import (
    transform_pdays,
    NUMERIC_COLS,
    ONEHOT_COLS,
    TARGET_COLS,
    CYCLICAL_COLS,
    BINARY_COLS
)

# Исходные хронологические данные UCI
df_temporal = pd.read_csv(
    project_root / 'data' / 'raw' / 'bank-additional-full.csv',
    sep=';'
)

# Данные из предыдущих экспериментов
df_original = pd.read_csv(
    project_root / 'data' / 'raw' / 'banking.csv'
)

print('Хронологический датасет:', df_temporal.shape)
print('Предыдущий датасет:', df_original.shape)

print('\nСтолбцы исходного датасета:')
print(df_temporal.columns.tolist())

print('\nРаспределение целевой переменной:')
print(df_temporal['y'].value_counts())

print('\nПервые пять наблюдений:')
display(df_temporal.head())


Хронологический датасет: (41188, 21)
Предыдущий датасет: (41188, 21)

Столбцы исходного датасета:
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']

Распределение целевой переменной:
y
no     36548
yes     4640
Name: count, dtype: int64

Первые пять наблюдений:


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 2. Проверка соответствия датасетов

Перед хронологическим разделением проверим, совпадают ли исходные наблюдения с данными предыдущих экспериментов.

Приведём названия признаков и целевую переменную к одинаковому формату, после чего сравним наборы строк независимо от их порядка.

При сравнении сохраним повторяющиеся строки: одинаковые наблюдения могут встречаться несколько раз.


In [2]:
# Приводим исходный датасет к формату banking.csv
df_temporal_clean = df_temporal.rename(
    columns=lambda col: col.replace('.', '_')
).copy()

df_temporal_clean['y'] = (
    df_temporal_clean['y'].map({'no': 0, 'yes': 1})
)

# Проверяем названия столбцов
print(
    'Совпадают ли наборы столбцов:',
    set(df_temporal_clean.columns) == set(df_original.columns)
)

# Одинаковый порядок столбцов
df_temporal_clean = df_temporal_clean[df_original.columns]

# Проверяем полное совпадение с учётом порядка строк
print(
    'Совпадают ли строки в исходном порядке:',
    df_temporal_clean.equals(df_original)
)

# Сравниваем наборы наблюдений независимо от порядка
# value_counts учитывает количество повторений каждой строки
temporal_counts = (
    df_temporal_clean.value_counts(dropna=False, sort=False)
    .sort_index()
)

original_counts = (
    df_original.value_counts(dropna=False, sort=False)
    .sort_index()
)

print(
    'Совпадают ли наблюдения независимо от порядка:',
    temporal_counts.equals(original_counts)
)

print(
    'Полных дубликатов в хронологическом датасете:',
    df_temporal_clean.duplicated().sum()
)

print(
    'Полных дубликатов в предыдущем датасете:',
    df_original.duplicated().sum()
)


Совпадают ли наборы столбцов: True
Совпадают ли строки в исходном порядке: False
Совпадают ли наблюдения независимо от порядка: True
Полных дубликатов в хронологическом датасете: 12
Полных дубликатов в предыдущем датасете: 12


## 3. Хронологическое разделение данных

Разделим наблюдения по исходному порядку: первые 80% используем для обучения, последние 20% — для тестирования.

В отличие от случайного стратифицированного разделения, здесь не перемешиваем строки и не выравниваем долю положительного класса между выборками.

Перед обучением сравним распределение целевой переменной в Train и Test. Это позволит оценить, изменяется ли частота положительных откликов между более ранними и поздними периодами.


In [3]:
# Граница хронологического разделения
split_idx = int(len(df_temporal_clean) * 0.8)

# Ранние наблюдения — Train
train_df = df_temporal_clean.iloc[:split_idx].copy()

# Поздние наблюдения — Test
test_df = df_temporal_clean.iloc[split_idx:].copy()

# Разделяем признаки и целевую переменную
X_train_time = train_df.drop(columns=['y', 'duration'])
y_train_time = train_df['y']

X_test_time = test_df.drop(columns=['y', 'duration'])
y_test_time = test_df['y']

print('Размер Train:', X_train_time.shape)
print('Размер Test:', X_test_time.shape)

# Сравниваем распределение целевой переменной
target_distribution = pd.DataFrame({
    'Train': y_train_time.value_counts(normalize=True),
    'Test': y_test_time.value_counts(normalize=True)
}).T

target_distribution.columns = ['Класс 0', 'Класс 1']

display((target_distribution * 100).round(2))

print(
    'Доля положительного класса в Train:',
    round(y_train_time.mean(), 4)
)

print(
    'Доля положительного класса в Test:',
    round(y_test_time.mean(), 4)
)


Размер Train: (32950, 19)
Размер Test: (8238, 19)


,Класс 0,Класс 1
Train,93.63,6.37
Test,69.17,30.83


Доля положительного класса в Train: 0.0637
Доля положительного класса в Test: 0.3083


### Вывод

Доля положительных откликов выросла с 6.37% на Train до 30.83% на Test. Из-за этого AP разных периодов нельзя напрямую сравнивать без учёта базового уровня.

## 4. Сравнение экономических показателей

В предыдущих экспериментах экономические признаки оказались наиболее важными для Logistic Regression.

Поскольку доля положительных откликов существенно отличается между ранним и поздним периодами, сравним распределения пяти экономических показателей.

Это позволит выявить возможные изменения распределений признаков (covariate shift).


In [4]:
economic_cols = [
    'emp_var_rate',
    'cons_price_idx',
    'cons_conf_idx',
    'euribor3m',
    'nr_employed'
]

# Статистики экономических признаков
economic_comparison = pd.DataFrame({
    'Train mean': X_train_time[economic_cols].mean(),
    'Test mean': X_test_time[economic_cols].mean(),
    'Train std': X_train_time[economic_cols].std(),
    'Test std': X_test_time[economic_cols].std()
})

# Разница между средними значениями
economic_comparison['Mean difference'] = (
    economic_comparison['Test mean']
    - economic_comparison['Train mean']
)

display(economic_comparison.round(3))


,Train mean,Test mean,Train std,Test std,Mean difference
emp_var_rate,0.650,-2.192,1.160,0.702,-2.842
cons_price_idx,93.704,93.062,0.481,0.649,-0.643
cons_conf_idx,-40.638,-39.963,3.867,6.851,0.675
euribor3m,4.269,1.031,1.284,0.243,-3.238
nr_employed,5195.140,5054.626,44.854,47.514,-140.515


### Вывод

Между периодами заметно изменились `emp_var_rate`, `euribor3m` и `nr_employed`. Поскольку эти признаки важны для модели, изменение их распределений может сказаться на качестве прогнозов.

## 5. Подготовка временной кросс-валидации

Для подбора гиперпараметров используем `TimeSeriesSplit` с четырьмя фолдами.

В отличие от `StratifiedKFold`, этот метод сохраняет исходный порядок наблюдений: обучение выполняется на прошлом, а валидация — на следующем временном интервале.

Перед обучением проверим размеры фолдов и долю положительного класса. Это особенно важно, поскольку распределение целевой переменной меняется со временем.

Хронологический Test остаётся полностью изолированным от подбора гиперпараметров.


In [5]:
# Временная кросс-валидация без перемешивания
time_cv = TimeSeriesSplit(n_splits=4)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(
    time_cv.split(X_train_time), start=1
):
    y_fold_train = y_train_time.iloc[train_idx]
    y_fold_val = y_train_time.iloc[val_idx]

    fold_results.append({
        'Fold': fold,
        'Train size': len(train_idx),
        'Validation size': len(val_idx),
        'Train positive %': y_fold_train.mean() * 100,
        'Validation positive %': y_fold_val.mean() * 100,
        'Train classes': y_fold_train.nunique(),
        'Validation classes': y_fold_val.nunique()
    })

fold_summary = pd.DataFrame(fold_results)

display(fold_summary.round(2))

# Проверяем, что в каждом фолде присутствуют оба класса
assert (fold_summary['Train classes'] == 2).all()
assert (fold_summary['Validation classes'] == 2).all()

print('Все фолды содержат оба класса.')


,Fold,Train size,Validation size,Train positive %,Validation positive %,Train classes,Validation classes
0,1,6590,6590,2.90,4.39,2,2
1,2,13180,6590,3.64,6.34,2,2
2,3,19770,6590,4.54,5.52,2,2
3,4,26360,6590,4.79,12.72,2,2


Все фолды содержат оба класса.


## 6. Обучение Logistic Regression

Используем Logistic Regression с One-Hot Encoding и стандартной нормализацией числовых признаков.

Все преобразования выполняются внутри `Pipeline`, чтобы параметры предобработки определялись только на обучающей части каждого фолда.

С помощью `GridSearchCV` подберём коэффициент регуляризации `C` и параметр `class_weight`, используя четыре последовательных временных фолда.

Основная метрика подбора — Average Precision. Дополнительно рассчитаем ROC-AUC.

После подбора лучшая конфигурация будет автоматически переобучена на всём хронологическом Train. Test пока не используем.


In [6]:
# Предобработка, аналогичная One-Hot модели
preprocessing = Pipeline([
    (
        'pdays',
        FunctionTransformer(
            transform_pdays,
            validate=False
        )
    ),
    (
        'columns',
        ColumnTransformer([
            ('numeric', StandardScaler(), NUMERIC_COLS),
            (
                'onehot',
                OneHotEncoder(
                    handle_unknown='ignore',
                    sparse_output=False
                ),
                ONEHOT_COLS + TARGET_COLS + CYCLICAL_COLS
            ),
            ('binary', 'passthrough', BINARY_COLS)
        ])
    )
])

# Новая, ещё не обученная модель
temporal_pipeline = Pipeline([
    ('preprocessing', preprocessing),
    (
        'model',
        LogisticRegression(max_iter=3000)
    )
])

In [7]:
# Сетка гиперпараметров
param_grid = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__class_weight': [None, 'balanced']
}

# Подбор только на хронологическом Train
temporal_grid = GridSearchCV(
    estimator=temporal_pipeline,
    param_grid=param_grid,
    scoring={
        'AP': 'average_precision',
        'ROC-AUC': 'roc_auc'
    },
    refit='AP',
    cv=time_cv,
    n_jobs=2,
    verbose=1
)

temporal_grid.fit(X_train_time, y_train_time)

print('Лучшие гиперпараметры:')
print(temporal_grid.best_params_)

print(
    '\nЛучший средний CV AP:',
    round(temporal_grid.best_score_, 4)
)

# Результаты всех конфигураций
cv_results_time = pd.DataFrame(
    temporal_grid.cv_results_
)

results_table = cv_results_time[[
    'param_model__C',
    'param_model__class_weight',
    'mean_test_AP',
    'std_test_AP',
    'mean_test_ROC-AUC'
]].copy()

results_table['param_model__class_weight'] = (
    results_table['param_model__class_weight']
    .fillna('None')
)

results_table = results_table.sort_values(
    'mean_test_AP',
    ascending=False
)

display(results_table.round(4))


Fitting 4 folds for each of 8 candidates, totalling 32 fits
Лучшие гиперпараметры:
{'model__C': 10, 'model__class_weight': 'balanced'}

Лучший средний CV AP: 0.076


,param_model__C,param_model__class_weight,mean_test_AP,std_test_AP,mean_test_ROC-AUC
7,10.00,balanced,0.0760,0.0372,0.5118
0,0.01,None,0.0748,0.0363,0.5118
2,0.10,None,0.0748,0.0358,0.5131
5,1.00,balanced,0.0746,0.0353,0.5140
6,10.00,None,0.0745,0.0361,0.5120
4,1.00,None,0.0744,0.0361,0.5116
3,0.10,balanced,0.0743,0.0357,0.5143
1,0.01,balanced,0.0741,0.0361,0.5125


## 7. Анализ качества на отдельных временных фолдах

Средний CV AP составил 0.0760, а ROC-AUC — 0.5118. Это указывает на слабую способность модели ранжировать наблюдения между временными периодами.

Поскольку доля положительного класса существенно различается между фолдами, отдельно сравним AP с базовым уровнем каждого валидационного периода.

Для этого используем уже рассчитанные результаты GridSearchCV, без повторного обучения моделей.


In [8]:
fold_diagnostics = []

# Индекс лучшей конфигурации GridSearchCV
best_idx = temporal_grid.best_index_

for fold, (train_idx, val_idx) in enumerate(
    time_cv.split(X_train_time), start=1
):
    # Базовый AP определяется долей положительного класса
    baseline_ap = y_train_time.iloc[val_idx].mean()

    # Метрики лучшей конфигурации на конкретном фолде
    fold_ap = temporal_grid.cv_results_[
        f'split{fold - 1}_test_AP'
    ][best_idx]

    fold_roc = temporal_grid.cv_results_[
        f'split{fold - 1}_test_ROC-AUC'
    ][best_idx]

    fold_diagnostics.append({
        'Fold': fold,
        'Baseline AP': baseline_ap,
        'Model AP': fold_ap,
        'AP Lift': fold_ap / baseline_ap,
        'ROC-AUC': fold_roc
    })

fold_diagnostics = pd.DataFrame(fold_diagnostics)

display(fold_diagnostics.round(4))


,Fold,Baseline AP,Model AP,AP Lift,ROC-AUC
0,1,0.0439,0.0430,0.9806,0.5100
1,2,0.0634,0.0701,1.1056,0.5424
2,3,0.0552,0.0525,0.9503,0.4395
3,4,0.1272,0.1382,1.0871,0.5554


### Вывод

На временных фолдах средний AP составил 0.0760 при базовом уровне около 0.0724, а ROC-AUC — 0.5118.

На первом и третьем фолдах AP оказалась ниже базового уровня. Худший ROC-AUC (0.4395) получен на третьем фолде.

Модель показала слабое и нестабильное качество при переносе между периодами.


## 8. Анализ изменения признаков между временными фолдами

Временная кросс-валидация показала низкое и нестабильное качество Logistic Regression.

Проверим, насколько распределения экономических признаков изменяются между обучающей и валидационной частями каждого фолда.

Для этого рассчитаем долю валидационных наблюдений, у которых значения экономических показателей выходят за минимальные и максимальные значения, встречавшиеся на соответствующем Train.

Дополнительно проверим, появляются ли на Validation месяцы, отсутствующие в обучающих данных.


In [9]:
shift_results = []

features_to_check = [
    'emp_var_rate',
    'euribor3m',
    'nr_employed'
]

for fold, (train_idx, val_idx) in enumerate(
    time_cv.split(X_train_time), start=1
):
    X_fold_train = X_train_time.iloc[train_idx]
    X_fold_val = X_train_time.iloc[val_idx]

    result = {'Fold': fold}

    # Проверяем экономические показатели
    for col in features_to_check:
        train_min = X_fold_train[col].min()
        train_max = X_fold_train[col].max()

        outside_range = (
            (X_fold_val[col] < train_min)
            | (X_fold_val[col] > train_max)
        )

        result[f'{col} outside %'] = (
            outside_range.mean() * 100
        )

    # Проверяем новые категории месяца
    unseen_months = (
        ~X_fold_val['month'].isin(X_fold_train['month'])
    )

    result['Unseen month %'] = (
        unseen_months.mean() * 100
    )

    shift_results.append(result)

shift_summary = pd.DataFrame(shift_results)

display(shift_summary.round(2))


,Fold,emp_var_rate outside %,euribor3m outside %,nr_employed outside %,Unseen month %
0,1,82.20,88.86,82.20,82.20
1,2,0.00,17.50,0.00,14.39
2,3,35.86,35.28,0.00,35.86
3,4,79.97,90.29,79.97,41.73


### Вывод

На первом и четвёртом фолдах более 80% значений `euribor3m` выходят за диапазон Train. Также на Validation встречаются месяцы, отсутствовавшие при обучении.

Это может затруднять прогнозирование, однако прямой зависимости между величиной сдвига признаков и качеством модели не обнаружено.



## 9. Оценка на хронологическом Test

Оценим лучшую Logistic Regression на последних 20% наблюдений, которые не использовались при обучении и подборе гиперпараметров.

Рассчитаем Average Precision, ROC-AUC и Precision@10%.

Поскольку доля положительного класса в Test значительно выше, чем в обучающем периоде, дополнительно сравним AP с базовым уровнем случайного ранжирования и рассчитаем Lift@10%.

Результаты позволят оценить, насколько модель способна ранжировать отклики в более позднем периоде.


In [10]:

from sklearn.metrics import average_precision_score, roc_auc_score

# Лучшая модель уже переобучена на всём хронологическом Train
best_temporal_model = temporal_grid.best_estimator_

# Вероятности положительного класса на Test
test_proba = best_temporal_model.predict_proba(X_test_time)[:, 1]

# Основные метрики
test_ap = average_precision_score(y_test_time, test_proba)
test_roc_auc = roc_auc_score(y_test_time, test_proba)

# Базовый AP
baseline_ap = y_test_time.mean()

# Выбираем 10% наблюдений с максимальной вероятностью
n_top = int(np.ceil(len(test_proba) * 0.10))
top_idx = np.argsort(-test_proba)[:n_top]

precision_at_10 = y_test_time.iloc[top_idx].mean()
lift_at_10 = precision_at_10 / baseline_ap

# Итоговая таблица
test_results = pd.DataFrame({
    'Метрика': [
        'Baseline AP',
        'Model AP',
        'AP Lift',
        'ROC-AUC',
        'Precision@10%',
        'Lift@10%'
    ],
    'Значение': [
        baseline_ap,
        test_ap,
        test_ap / baseline_ap,
        test_roc_auc,
        precision_at_10,
        lift_at_10
    ]
})

display(test_results.round(4))

print('Контактов в топ-10%:', n_top)
print(
    'Положительных откликов в топ-10%:',
    int(y_test_time.iloc[top_idx].sum())
)


,Метрика,Значение
0,Baseline AP,0.3083
1,Model AP,0.4905
2,AP Lift,1.5907
3,ROC-AUC,0.6719
4,Precision@10%,0.6226
5,Lift@10%,2.0192


Контактов в топ-10%: 824
Положительных откликов в топ-10%: 513


## 10. Устойчивость качества внутри Test

Разделим хронологический Test на две последовательные части и отдельно рассчитаем метрики для каждой.

Модель и гиперпараметры не меняем. Проверка носит диагностический характер и не используется для выбора новой конфигурации.


In [11]:

test_middle = len(y_test_time) // 2

periods = {
    'Первая половина': slice(0, test_middle),
    'Вторая половина': slice(test_middle, None)
}

period_results = []

for period, idx in periods.items():
    y_part = y_test_time.iloc[idx].to_numpy()
    proba_part = test_proba[idx]

    baseline = y_part.mean()
    ap = average_precision_score(y_part, proba_part)
    roc = roc_auc_score(y_part, proba_part)

    n_top = int(np.ceil(len(y_part) * 0.10))
    top_idx = np.argsort(-proba_part)[:n_top]

    precision = y_part[top_idx].mean()

    period_results.append({
        'Период': period,
        'Наблюдений': len(y_part),
        'Positive rate': baseline,
        'AP': ap,
        'AP / Baseline': ap / baseline,
        'ROC-AUC': roc,
        'Precision@10%': precision,
        'Lift@10%': precision / baseline
    })

period_comparison = pd.DataFrame(period_results)

display(period_comparison.round(4))


,Период,Наблюдений,Positive rate,AP,AP / Baseline,ROC-AUC,Precision@10%,Lift@10%
0,Первая половина,4119,0.1573,0.3295,2.0943,0.6772,0.4369,2.7771
1,Вторая половина,4119,0.4593,0.5678,1.2362,0.6271,0.6165,1.3422


## 11. Итоги

Хронологическая валидация показала, что качество Logistic Regression заметно зависит от периода.

На четырёх временных фолдах средний ROC-AUC составил 0.5118, а AP — 0.0760 при базовом уровне около 0.0724. Модель практически не превосходила случайное ранжирование.

На финальном Test результаты оказались лучше: ROC-AUC = 0.6719, AP = 0.4905 и Lift@10% = 2.02.

При разделении Test на две последовательные части выяснилось, что ROC-AUC снизился с 0.6772 до 0.6271, а Lift@10% — с 2.78 до 1.34. При этом доля положительных откликов выросла с 15.73% до 45.93%.

Между периодами также заметно изменились экономические показатели. Это может частично объяснять нестабильность результатов, но установить точные причины по текущим экспериментам нельзя.

### Ограничения

- Разделение выполнено по исходному порядку строк, без точных дат контактов.
- Test не использовался для обучения и подбора гиперпараметров, но его распределения предварительно изучались.
- Различия в доле положительного класса затрудняют прямое сравнение Average Precision между периодами.

Модель показала полезное ранжирование на последнем периоде, однако результаты временной кросс-валидации и изменения качества внутри Test не позволяют считать её устойчивой к изменениям условий.
